In [13]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from statsmodels.stats.weightstats import ttost_paired

In [14]:
# ==============================================================================
# STEP 1: WILDFIRE EXPOSURE SUBINDEX (CLIMATE + LAND COVER + ELEVATION)
# ==============================================================================
print("\n--- [1/4] Processing Wildfire Exposure Subindex ---")

# 1.1 Load Raw Datasets
climate = pd.read_csv('../../data/climate.csv')
land = pd.read_csv('../../data/nhgis_county2022_tl2022_nlcd2021.csv')
elev = pd.read_csv('../../data/elevacion_media.csv')
wrc = pd.read_excel('../../data/wrc_202505.xlsx', sheet_name='Counties')

# Standardize GEOID key formats (5-digit padded strings)
wrc['GEOID'] = wrc['GEOID'].astype(str).str.zfill(5)
wrc = wrc.set_index('GEOID').dropna()

land.columns = land.columns.str.lower()
land['geoid'] = land['geoid'].astype(str).str.zfill(5)

climate['geoid'] = climate['geoid'].astype(str).str.zfill(5)
climate['week_end'] = pd.to_datetime(climate['week_end'])
climate = climate.sort_values(by=['geoid', 'week_end'])

elev['geoid'] = elev['geoid'].astype(str).str.zfill(5)
elev = elev.set_index('geoid')

# 1.2 Pivot Time Series & Compute Summary Statistics
temp_wide = climate.pivot(index='geoid', columns='week_end', values='t_mean')
rh_wide = climate.pivot(index='geoid', columns='week_end', values='rh_mean')
drought_wide = climate.pivot(index='geoid', columns='week_end', values='dsci')

wind_max = climate.groupby('geoid')['wind_speed_mean'].quantile(0.9).to_frame(name='wind_max')
wind_gusts_max = climate.groupby('geoid')['wind_gusts_max'].quantile(0.9).to_frame(name='wind_gusts_max')
precip_tot = climate.groupby('geoid')['precip_sum'].sum().to_frame(name='precip_tot') / 3

# 1.3 Group Land Cover into 7 Representative Classes
land_cols = ['geoid', 'prop_11', 'prop_12', 'prop_21', 'prop_22', 'prop_23', 'prop_24',
             'prop_31', 'prop_41', 'prop_42', 'prop_43', 'prop_52', 'prop_71', 'prop_81',
             'prop_82', 'prop_90', 'prop_95']
land_filtered = land[land_cols].set_index('geoid')

land_grouped = pd.DataFrame(index=land_filtered.index)
land_grouped['urban_land'] = land_filtered[['prop_21', 'prop_22', 'prop_23', 'prop_24']].sum(axis=1)
land_grouped['forest_land'] = land_filtered[['prop_41', 'prop_42', 'prop_43']].sum(axis=1)
land_grouped['agricultural_land'] = land_filtered[['prop_81', 'prop_82']].sum(axis=1)
land_grouped['wetland'] = land_filtered[['prop_90', 'prop_95']].sum(axis=1)
land_grouped['shrubland'] = land_filtered['prop_52']
land_grouped['grassland'] = land_filtered['prop_71']
land_grouped['non_vegetated'] = land_filtered[['prop_11', 'prop_12', 'prop_31']].sum(axis=1)

# 1.4 Synchronize Base GEOIDs
common_geoids_exp = (
    temp_wide.index
    .intersection(elev.index)
    .intersection(drought_wide.index)
    .intersection(wrc.index)
)

temp_wide = temp_wide.loc[common_geoids_exp]
rh_wide = rh_wide.loc[common_geoids_exp]
drought_wide = drought_wide.loc[common_geoids_exp]
wind_max = wind_max.loc[common_geoids_exp]
wind_gusts_max = wind_gusts_max.loc[common_geoids_exp]
precip_tot = precip_tot.loc[common_geoids_exp]
elev_sub = elev.loc[common_geoids_exp, ['mean_elev']]

# 1.5 Isolated Dimensionality Reduction via PCA (90% Variance Threshold)
def reduce_pca(df, prefix, threshold=0.90):
    scaled = StandardScaler().fit_transform(df)
    n_comp = np.argmax(np.cumsum(PCA().fit(scaled).explained_variance_ratio_) >= threshold) + 1
    pca_res = PCA(n_components=n_comp).fit_transform(scaled)
    cols = [f'{prefix}_pca_{i+1}' for i in range(n_comp)]
    return pd.DataFrame(pca_res, index=df.index, columns=cols)

df_temp_pca = reduce_pca(temp_wide, 'temp')
df_rh_pca = reduce_pca(rh_wide, 'rh')
df_drought_pca = reduce_pca(drought_wide, 'drought')

# Assemble Climate + Elevation Features
climate_elev_features = pd.concat([
    df_temp_pca, df_rh_pca, df_drought_pca, 
    precip_tot, wind_max, wind_gusts_max, elev_sub
], axis=1)

# Separate CONUS vs. Alaska GEOIDs (Alaska FIPS starts with '02')
all_geoids = climate_elev_features.index
alaska_geoids = all_geoids[all_geoids.str.startswith('02')]
conus_geoids = all_geoids[~all_geoids.str.startswith('02')].intersection(land_grouped.index)

# CONUS Feature Matrix (Climate + Land Cover + Elevation)
conus_features_df = pd.concat([climate_elev_features.loc[conus_geoids], land_grouped.loc[conus_geoids]], axis=1).dropna()
conus_geoids = conus_features_df.index
y_conus = wrc.loc[conus_geoids, 'BP_NATIONAL_RANK']

# Alaska Feature Matrix (Climate + Elevation only, excluding Land Cover)
alaska_features_df = climate_elev_features.loc[alaska_geoids].dropna()
alaska_geoids = alaska_features_df.index
y_alaska = wrc.loc[alaska_geoids, 'BP_NATIONAL_RANK']



--- [1/4] Processing Wildfire Exposure Subindex ---


In [15]:
# ==============================================================================
# HELPER FUNCTIONS: ARCHITECTURES & MODEL SELECTION ENGINE
# ==============================================================================

def build_custom_nn(input_shape):
    model = models.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Dense(128, kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('leaky_relu'),
        layers.Dropout(0.2),
        layers.Dense(64, kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('leaky_relu'),
        layers.Dense(32, activation='leaky_relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizers.Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])
    return model

def build_wide_and_deep_nn(input_shape):
    inputs = layers.Input(shape=(input_shape,))
    
    # Wide path (linear memorization)
    wide = layers.Dense(1, activation='linear')(inputs)
    
    # Deep path (nonlinear generalization)
    deep = layers.Dense(128, kernel_initializer='he_normal')(inputs)
    deep = layers.BatchNormalization()(deep)
    deep = layers.Activation('leaky_relu')(deep)
    deep = layers.Dropout(0.2)(deep)
    deep = layers.Dense(64, kernel_initializer='he_normal')(deep)
    deep = layers.BatchNormalization()(deep)
    deep = layers.Activation('leaky_relu')(deep)
    deep = layers.Dense(32, activation='leaky_relu')(deep)
    deep_out = layers.Dense(1, activation='linear')(deep)
    
    # Combine paths
    combined = layers.add([wide, deep_out])
    outputs = layers.Activation('sigmoid')(combined)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])
    return model

def evaluate_candidate(y_true, y_pred, eps=0.05):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    tost_res = ttost_paired(y_pred, y_true, -eps, eps)
    p_val = tost_res[0]
    passes_tost = (p_val < 0.05)
    return passes_tost, rmse, p_val

def train_and_select_model(X_raw, y_target, eps=0.05, region_label="CONUS"):
    print(f"\n--- Model Benchmark & Selection: {region_label} (N = {len(X_raw)}) ---")
    
    # Standardize input features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    input_dim = X_scaled.shape[1]
    y_values = y_target.values
    
    candidates = {}
    
    # 1. Linear Regression
    lr = LinearRegression()
    lr.fit(X_scaled, y_values)
    preds_lr = np.clip(lr.predict(X_scaled), 0, 1)
    pass_lr, rmse_lr, p_lr = evaluate_candidate(y_values, preds_lr, eps)
    candidates['Linear Regression'] = {'preds': preds_lr, 'rmse': rmse_lr, 'pass_tost': pass_lr, 'p_val': p_lr}
    
    # 2. XGBoost Regressor
    xgb_mod = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5)
    xgb_mod.fit(X_scaled, y_values)
    preds_xgb = np.clip(xgb_mod.predict(X_scaled), 0, 1)
    pass_xgb, rmse_xgb, p_xgb = evaluate_candidate(y_values, preds_xgb, eps)
    candidates['XGBoost'] = {'preds': preds_xgb, 'rmse': rmse_xgb, 'pass_tost': pass_xgb, 'p_val': p_xgb}
    
    # 3. Custom Neural Network
    nn_custom = build_custom_nn(input_dim)
    nn_custom.fit(X_scaled, y_values, epochs=150, batch_size=8, verbose=0)
    preds_nn = nn_custom.predict(X_scaled, verbose=0).flatten()
    preds_nn = np.clip(preds_nn, 0, 1)
    pass_nn, rmse_nn, p_nn = evaluate_candidate(y_values, preds_nn, eps)
    candidates['Custom NN'] = {'preds': preds_nn, 'rmse': rmse_nn, 'pass_tost': pass_nn, 'p_val': p_nn}
    
    # 4. Wide & Deep Neural Network
    nn_wd = build_wide_and_deep_nn(input_dim)
    nn_wd.fit(X_scaled, y_values, epochs=150, batch_size=8, verbose=0)
    preds_wd = nn_wd.predict(X_scaled, verbose=0).flatten()
    preds_wd = np.clip(preds_wd, 0, 1)
    pass_wd, rmse_wd, p_wd = evaluate_candidate(y_values, preds_wd, eps)
    candidates['Wide & Deep NN'] = {'preds': preds_wd, 'rmse': rmse_wd, 'pass_tost': pass_wd, 'p_val': p_wd}
    
    # Print Benchmark Table
    print(f"{'Model Architecture':<22} | {'TOST Pass (p<0.05)':<20} | {'p-value':<12} | {'RMSE':<10}")
    print("-" * 72)
    for name, res in candidates.items():
        print(f"{name:<22} | {str(res['pass_tost']):<20} | {res['p_val']:.4e}   | {res['rmse']:.4f}")
    
    # Selection Rule: Must pass TOST (p < 0.05), then select lowest RMSE
    passed_models = {k: v for k, v in candidates.items() if v['pass_tost']}
    
    if passed_models:
        best_name = min(passed_models, key=lambda k: passed_models[k]['rmse'])
        print(f"\n✅ Selected Model for {region_label}: '{best_name}' (Passed TOST, Lowest RMSE: {passed_models[best_name]['rmse']:.4f})")
        return candidates[best_name]['preds']
    else:
        best_name = min(candidates, key=lambda k: candidates[k]['rmse'])
        print(f"\n⚠️ Fallback for {region_label}: No model passed strict TOST (eps={eps}). Selected lowest RMSE model: '{best_name}' ({candidates[best_name]['rmse']:.4f})")
        return candidates[best_name]['preds']


In [17]:
# ==============================================================================
# 1.6 RUN SELECTION AND ASSEMBLE EXPOSURE SUBINDEX
# ==============================================================================


# Train and predict for CONUS counties
conus_preds = train_and_select_model(conus_features_df, y_conus, eps=0.05, region_label="CONUS")
df_exp_conus = pd.DataFrame({'Exposure_Subindex': conus_preds}, index=conus_geoids)

# Train and predict for Alaska boroughs (excluding land cover)
if len(alaska_features_df) > 0:
    alaska_preds = train_and_select_model(alaska_features_df, y_alaska, eps=0.05, region_label="Alaska")
    df_exp_alaska = pd.DataFrame({'Exposure_Subindex': alaska_preds}, index=alaska_geoids)
    df_exposure = pd.concat([df_exp_conus, df_exp_alaska], axis=0)
else:
    df_exposure = df_exp_conus

print(f"\n✅ Wildfire Exposure Subindex successfully constructed for {len(df_exposure)} total counties/boroughs.")


--- Model Benchmark & Selection: CONUS (N = 3099) ---
Model Architecture     | TOST Pass (p<0.05)   | p-value      | RMSE      
------------------------------------------------------------------------
Linear Regression      | True                 | 1.0165e-70   | 0.1516
XGBoost                | True                 | 9.1496e-309   | 0.0659
Custom NN              | True                 | 5.6316e-266   | 0.0622
Wide & Deep NN         | True                 | 0.0000e+00   | 0.0562

✅ Selected Model for CONUS: 'Wide & Deep NN' (Passed TOST, Lowest RMSE: 0.0562)

--- Model Benchmark & Selection: Alaska (N = 27) ---
Model Architecture     | TOST Pass (p<0.05)   | p-value      | RMSE      
------------------------------------------------------------------------
Linear Regression      | True                 | 0.0000e+00   | 0.0000
XGBoost                | True                 | 1.3330e-22   | 0.0079
Custom NN              | True                 | 9.6535e-04   | 0.0602
Wide & Deep NN         |

In [18]:
# ==============================================================================
# STEP 2: ECONOMIC FRAGILITY SUBINDEX
# ==============================================================================
print("\n--- [2/4] Processing Economic Fragility Subindex ---")

econ = pd.read_csv("../../data/economic_clusters_labeled.csv")
econ['geoid'] = econ['FIPS'].astype(str).str.zfill(5)

variables_econ_pca = ['Poverty_Rate', 'Household_Income', 'RUCC_2023']
X_econ = econ[variables_econ_pca].dropna()

scaler_econ = StandardScaler()
X_econ_scaled = scaler_econ.fit_transform(X_econ)

pca_econ = PCA(n_components=1)
pc1_econ = pca_econ.fit_transform(X_econ_scaled)

minmax_econ = MinMaxScaler(feature_range=(0, 1))
score_econ_raw = minmax_econ.fit_transform(pc1_econ).flatten()

# Direction Alignment: Ensure 1 = Highly Vulnerable (Positive correlation with Poverty Rate)
corr_poverty = pd.Series(score_econ_raw).corr(econ.loc[X_econ.index, 'Poverty_Rate'])
if corr_poverty < 0:
    score_econ = 1.0 - score_econ_raw
else:
    score_econ = score_econ_raw

var_econ = pca_econ.explained_variance_ratio_[0]
print(f"Economic PC1 explained variance: {var_econ:.2%}")

df_economic = pd.DataFrame({'Economic_Subindex': score_econ}, index=econ.loc[X_econ.index, 'geoid'])


--- [2/4] Processing Economic Fragility Subindex ---
Economic PC1 explained variance: 68.12%


In [21]:
# ==============================================================================
# STEP 3: SOCIODEMOGRAPHIC VULNERABILITY SUBINDEX
# ==============================================================================
print("\n--- [3/4] Processing Sociodemographic Vulnerability Subindex ---")

import us

# 3.1 Load Demographics Dataset
demog = pd.read_excel("../../data/Clusters_Finales.xlsx", sheet_name="Unnormalized w.reduced vars")

# 3.2 Fetch and Format Census FIPS Lookup Table
fips_url = 'https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt'
fips_df = pd.read_csv(
    fips_url, 
    header=None, 
    names=['state', 'state_fips', 'county_fips', 'county_name', 'class code'], 
    dtype=str
)

fips_df['fips'] = fips_df['state_fips'] + fips_df['county_fips']
fips_df['county'] = fips_df['county_name'].str.strip().str.lower()

# Convert state abbreviation to full lowercase name (e.g., 'AL' -> 'alabama')
fips_df['state'] = fips_df['state'].apply(
    lambda x: us.states.lookup(x).name.lower() if us.states.lookup(x) else None
)

# 3.3 Clean and Standardize Strings in Demographics Data for Joining
demog['county'] = demog['county'].astype(str).str.strip().str.lower()

# Handle whether 'state' in demog is an abbreviation ('CA') or full name ('California')
demog['state'] = demog['state'].apply(
    lambda x: us.states.lookup(str(x)).name.lower() if us.states.lookup(str(x)) else str(x).strip().str.lower()
)

# 3.4 Merge to Attach 5-Digit GEOID ('fips')
demog_merged = pd.merge(
    fips_df[['fips', 'county', 'state']], 
    demog, 
    how='inner',
    left_on=['county', 'state'], 
    right_on=['county', 'state']
)

demog_merged['geoid'] = demog_merged['fips'].astype(str).str.zfill(5)

# 3.5 Select sociodemographic variables
variables_demo = [
    'disability_prevalence',
    'language_otherenglish',
    'education_bachelorsorhigher',
    'minority_share',
    'dependency_ratio',
    'without_insuranceunder65'
]

# Set GEOID as index and drop rows with missing values
X_demo_df = demog_merged.set_index('geoid')[variables_demo].dropna()

# 3.6 Scale variables to [0,1]
scaler_demo = MinMaxScaler()

X_demo_scaled = pd.DataFrame(
    scaler_demo.fit_transform(X_demo_df),
    columns=X_demo_df.columns,
    index=X_demo_df.index
)

# 3.7 Reverse education direction:
# higher education = lower vulnerability
X_demo_scaled['education_bachelorsorhigher'] = (
    1 - X_demo_scaled['education_bachelorsorhigher']
)

# 3.8 Compute composite sociodemographic vulnerability score
score_demo = X_demo_scaled.mean(axis=1)

# 3.9 Assemble final subindex output
df_demographic = pd.DataFrame(
    {'Demographic_Subindex': score_demo},
    index=X_demo_df.index
)

print(
    f"Sociodemographic Subindex successfully created "
    f"for {len(df_demographic)} counties."
)


--- [3/4] Processing Sociodemographic Vulnerability Subindex ---
Sociodemographic Subindex successfully created for 3119 counties.


In [22]:
# ==============================================================================
# STEP 4: W-RVI FINAL INTEGRATION (Deduplicated Fix)
# ==============================================================================
print("\n--- [4/4] Integrating Subindices into W-RVI ---")

# Ensure index uniqueness across all three subindices
df_exposure = df_exposure[~df_exposure.index.duplicated(keep='first')]
df_economic = df_economic[~df_economic.index.duplicated(keep='first')]
df_demographic = df_demographic[~df_demographic.index.duplicated(keep='first')]

# Find common unique GEOIDs
final_geoids = (
    df_exposure.index
    .intersection(df_economic.index)
    .intersection(df_demographic.index)
    .unique()
)

# Align and concatenate cleanly along columns (axis=1)
wrvi_df = pd.concat([
    df_exposure.loc[final_geoids],
    df_economic.loc[final_geoids],
    df_demographic.loc[final_geoids]
], axis=1)

# Apply IPCC / WorldRiskIndex weights
WEIGHT_EXPOSURE = 0.50
WEIGHT_ECONOMIC = 0.35
WEIGHT_DEMOGRAPHIC = 0.15

wrvi_df['W_RVI'] = (
    (WEIGHT_EXPOSURE * wrvi_df['Exposure_Subindex']) +
    (WEIGHT_ECONOMIC * wrvi_df['Economic_Subindex']) +
    (WEIGHT_DEMOGRAPHIC * wrvi_df['Demographic_Subindex'])
)

print(f"W-RVI successfully computed for {len(wrvi_df)} unique counties.")
print(wrvi_df.head())


--- [4/4] Integrating Subindices into W-RVI ---
W-RVI successfully computed for 3112 unique counties.
       Exposure_Subindex  Economic_Subindex  Demographic_Subindex     W_RVI
01001           0.545678           0.395969              0.320667  0.459529
01003           0.817399           0.375510              0.297440  0.584744
01005           0.593359           0.687675              0.446230  0.604300
01007           0.670491           0.519651              0.379898  0.574108
01009           0.731922           0.401465              0.360035  0.560479


In [23]:
wrvi_df.to_csv("../../data/wrvi_index.csv", index_label='geoid')

### Reformat columns

In [24]:
import pandas as pd

# 1. Cargar los DataFrames
wrvi_df = pd.read_csv('../../data/wrvi_index.csv')
climate_cluster_df = pd.read_csv('../../data/climate_cluster.csv')

# 2. Convertir la columna geoid a string de 5 dígitos con ceros a la izquierda
wrvi_df['geoid'] = wrvi_df['geoid'].astype(str).str.zfill(5)
climate_cluster_df['geoid'] = climate_cluster_df['geoid'].astype(str).str.zfill(5)

# 3. Traer state_name y county_name desde climate_cluster_df
wrvi_df = wrvi_df.merge(
    climate_cluster_df[['geoid', 'state_name', 'county_name']],
    on='geoid',
    how='left'
)

# 4. Reordenar las columnas en el orden especificado
columnas_deseadas = [
    'geoid', 
    'state_name', 
    'county_name', 
    'Exposure_Subindex', 
    'Economic_Subindex', 
    'Demographic_Subindex', 
    'W_RVI'
]
wrvi_df = wrvi_df[columnas_deseadas]

# 5. Convertir todos los nombres de columnas a minúsculas
wrvi_df.columns = wrvi_df.columns.str.lower()

# Verificar resultado
print(wrvi_df.head())

   geoid state_name     county_name  exposure_subindex  economic_subindex  \
0  01001    alabama  autauga county           0.545678           0.395969   
1  01003    alabama  baldwin county           0.817399           0.375510   
2  01005    alabama  barbour county           0.593359           0.687675   
3  01007    alabama     bibb county           0.670491           0.519651   
4  01009    alabama   blount county           0.731922           0.401465   

   demographic_subindex     w_rvi  
0              0.320667  0.459529  
1              0.297440  0.584744  
2              0.446230  0.604300  
3              0.379898  0.574108  
4              0.360035  0.560479  


In [26]:
wrvi_df.to_csv('../../data/wrvi_index.csv', index=False)

## Re-scale subindexes, calculate W-RVI, and sensitivity analysis

In [1]:
import pandas as pd

In [13]:
wrvi = pd.read_csv('../../data/wrvi_index.csv')

In [14]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
wrvi['exposure_subindex'] = scaler.fit_transform(wrvi[['exposure_subindex']])
wrvi['economic_subindex'] = scaler.fit_transform(wrvi[['economic_subindex']])
wrvi['demographic_subindex'] = scaler.fit_transform(
    wrvi[['demographic_subindex']]
)

In [15]:
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# Define vulnerability weight splits (Econ / Dem) within the 50% vulnerability allocation
# Total weight formula: 0.50 * Exposure + 0.50 * (w_econ * Econ + w_dem * Dem)
schemes = {
    '50/50': (0.25, 0.25),
    '60/40': (0.30, 0.20),
    '70/30': (0.35, 0.15),
    '80/20': (0.40, 0.10),
}

# Compute W-RVI for each weighting scheme
for name, (w_econ, w_dem) in schemes.items():
    wrvi[f'wrvi_{name}'] = (
        0.50 * wrvi['exposure_subindex']
        + w_econ * wrvi['economic_subindex']
        + w_dem * wrvi['demographic_subindex']
    )

# Rank correlation against baseline (70/30)
baseline_split = '70/30'
baseline_col = f'wrvi_{baseline_split}'
results = []

for name in schemes.keys():
    if name == baseline_split:
        continue
    target_col = f'wrvi_{name}'
    rho, _ = spearmanr(wrvi[baseline_col], wrvi[target_col])
    tau, _ = kendalltau(wrvi[baseline_col], wrvi[target_col])
    results.append({
        'Vulnerability Split (Econ/Dem)': name,
        'Econ Total Weight': schemes[name][0],
        'Dem Total Weight': schemes[name][1],
        "Spearman's Rho (ρ)": round(rho, 4),
        "Kendall's Tau (τ)": round(tau, 4),
    })

sensitivity_results = pd.DataFrame(results)
print(sensitivity_results.to_string(index=False))

Vulnerability Split (Econ/Dem)  Econ Total Weight  Dem Total Weight  Spearman's Rho (ρ)  Kendall's Tau (τ)
                         50/50               0.25              0.25              0.9984             0.9648
                         60/40               0.30              0.20              0.9996             0.9823
                         80/20               0.40              0.10              0.9996             0.9823


In [ ]:
# Apply weights
WEIGHT_EXPOSURE = 0.50
WEIGHT_ECONOMIC = 0.35
WEIGHT_DEMOGRAPHIC = 0.15

wrvi['w_rvi'] = (
    (WEIGHT_EXPOSURE * wrvi['exposure_subindex']) +
    (WEIGHT_ECONOMIC * wrvi['economic_subindex']) +
    (WEIGHT_DEMOGRAPHIC * wrvi['demographic_subindex'])
)

wrvi = wrvi.drop(columns=['wrvi_50/50', 'wrvi_70/30', 'wrvi_60/40', 'wrvi_80/20'])

In [17]:
wrvi.to_csv('../../data/wrvi_index.csv', index=False)